# Semiconductor Market & Supply Chain Sentiment Analysis

The semiconductor industry is highly cyclical and prone to extreme bottlenecks. Companies need to predict supply chain disruptions before they happen. The project wants to discuss how global supply chain delays affect AI advancement. It shows how NLP sentiment engine detected early warnings of hardware shortages weeks before they hit mainstream financial news. 

## Install libraries

In [0]:
%pip install edgartools yfinance transformers torch 
dbutils.library.restartPython # after instaling library, restart python

## Libraries

In [0]:

from edgar import Company, set_identity
from transformers import pipeline
import yfinance as yf

import pandas as pd
import numpy as np

## SEC EDGAR Dataset Loading 

reference : [SEC API.io](https://sec-api.io/docs/sec-filings-item-extraction-api/python-example)  
reference : [edgartools 3.13.4](https://pypi.org/project/edgartools/3.13.4/)

By using SEC EDGAR API key, we want to load company datasets for NVIDIA (NVDA), TSMC (TSM), ASML (ASML), AMD (AMD), and Micron (MU).

For each dataset, we want to extract 4 columns, including company name (ticker, company identifier), filing_date, form_type (annual 10-K, 20-F), item_1a_text (text, Risk Factors). 

Form 10-K is for U.S. companies, such as NVDA, AMD, and MU.   
Form 20-F is for foreign companies, such as TSMC, ASML.   

In [0]:
set_identity("Hyeonsuk Park hyeonsuk.park@hotmail.com")

tickers = ["NVDA", "TSM", "ASML", "AMD", "MU"]

companies_data = []

for ticker in tickers:
    try:
        print(f"Fetching 10-k for {ticker}...")
        company = Company(ticker)

        # Get the latest 10-k filing
        filings = company.get_filings(form="10-K")

        # if empty, fall back to 20-F
        if len(filings) == 0:
            print(f" -> {ticker} is a foreign issuer. Fetching Form 20-F...")
            filings = company.get_filings(form="20-F")

        latest_filing = filings.latest()

        try:
            # attempt to pull item 1A / Risk Factors dynamically
            risk_text = latest_filing.obj().risk_factors
        except Exception:
            risk_text = None

        if not risk_text:
            risk_text = latest_filing.text()

        companies_data.append({
            "ticker": ticker,
            "company_name": company.name,
            "filing_date": str(latest_filing.filing_date),
            "form": latest_filing.form,
            "risk_factors_text": str(risk_text)[:10000] 

        })

    except Exception as e:
        print(f"Failed to process {ticker}: {e}")


Issue: only US companies returend data, with 10-K. foreign companies use 20-F.

In [0]:
# Convert to Pandas DataFrame
df_raw = pd.DataFrame(companies_data)

# Convert to PySpark DataFrame for Bronze Delta Lake
df_bronze = spark.createDataFrame(df_raw)

# Display to vertify columns
display(df_bronze.select("ticker", "filing_date", "company_name", "form", "risk_factors_text"))

## Load Stock dataset from Yahoo Finance